# DX 704 Week 10 Project

In this project, you will implement document search within a question and answer database and assess its performance.


The full project description and a template notebook are available on GitHub: [Project 10 Materials](https://github.com/bu-cds-dx704/dx704-project-10).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download the SQuAD-explorer Data Set

You may use the code provided below.

In [1]:
!git clone https://github.com/rajpurkar/SQuAD-explorer

fatal: destination path 'SQuAD-explorer' already exists and is not an empty directory.


In [2]:
import json

In [3]:
with open("SQuAD-explorer/dataset/train-v1.1.json") as fp:
    train_data = json.load(fp)

In [4]:
type(train_data)

dict

In [5]:
list(train_data.keys())

['data', 'version']

In [6]:
type(train_data["data"])

list

In [7]:
len(train_data["data"])

442

In [8]:
type(train_data["data"][0])

dict

In [9]:
train_data["data"][0].keys()

dict_keys(['title', 'paragraphs'])

In [10]:
train_data["data"][0]["title"]

'University_of_Notre_Dame'

In [11]:
len(train_data["data"][0]["paragraphs"])

55

In [12]:
train_data["data"][0]["paragraphs"][0]

{'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'qas': [{'answers': [{'answer_start': 515,
     'text': 'Saint Bernadette Soubirous'}],
   'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
   'id': '5733be284776f41900661182'},
  {'answers': [{'answer_start': 188, 'text': 'a copper statue of Christ

In [13]:
sum(len(doc["paragraphs"]) for doc in train_data["data"])

18896

## Part 2: Restructure JSON Data for Processing

Parse the file "SQuAD-explorer/dataset/train-v1.1.json" above to produce a file "parsed.tsv" with columns document_title, paragraph_index, and paragraph_context.
The paragraph_index column should be zero-indexed, so zero for the first paragraph of each document.
Use pandas `to_csv` method to write the file since there are many quotes and other issues to handle otherwise.

In [14]:
# YOUR CHANGES HERE

import pandas as pd

# One row per paragraph; enumerate restarts at zero for each document.
parsed_rows = [
    {
        "document_title": doc["title"],
        "paragraph_index": paragraph_index,
        "paragraph_context": paragraph["context"],
    }
    for doc in train_data["data"]
    for paragraph_index, paragraph in enumerate(doc["paragraphs"])
]

parsed_df = pd.DataFrame(
    parsed_rows,
    columns=["document_title", "paragraph_index", "paragraph_context"],
)

parsed_df.to_csv("parsed.tsv", sep="\t", index=False)

parsed_df.head()

,document_title,paragraph_index,paragraph_context
0,University_of_Notre_Dame,0,"Architecturally, the school has a Catholic cha..."
1,University_of_Notre_Dame,1,"As at most other universities, Notre Dame's st..."
2,University_of_Notre_Dame,2,The university is the major seat of the Congre...
3,University_of_Notre_Dame,3,The College of Engineering was established in ...
4,University_of_Notre_Dame,4,All of Notre Dame's undergraduate students are...


Submit "parsed.tsv" in Gradescope.

## Part 3: Prepare Suitable Paragraph Vectors for Document Search

Design and implement paragraph vectors based on their text with length 1024.
Note that this will be much smaller than the number of distinct words in the training data.

Hint: you can base your vectors on any techniques covered in this module so far.
Beware that they will be automatically assessed (along with the question vectors of part 4) to make sure they retain useful information.

In [15]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.preprocessing import normalize

parsed_df = pd.read_csv("parsed.tsv", sep="\t")

# Hashing fixes the length at 1024 without storing a vocabulary; questions reuse these objects in part 4.
hasher = HashingVectorizer(n_features=1024, stop_words="english", alternate_sign=False, norm=None)
tfidf = TfidfTransformer(sublinear_tf=True)

paragraph_counts = hasher.transform(parsed_df["paragraph_context"])
paragraph_vectors = normalize(tfidf.fit_transform(paragraph_counts)).toarray()
paragraph_vectors = np.round(paragraph_vectors, 6)

paragraph_vectors.shape

(18896, 1024)

Save your paragraph vectors in a file "paragraph-vectors.tsv.gz" with columns document_title, paragraph_index, and paragraph_vector_json where paragraph_vector_json is a JSON encoded list.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

In [16]:
import json

paragraph_vectors_df = parsed_df[["document_title", "paragraph_index"]].copy()
paragraph_vectors_df["paragraph_vector_json"] = [json.dumps(v.tolist()) for v in paragraph_vectors]
paragraph_vectors_df.to_csv("paragraph-vectors.tsv.gz", sep="\t", index=False)

Submit "paragraph-vectors.tsv.gz" in Gradescope.

## Part 4: Encode Question Vectors with the Same Design

Read the questions in "questions.tsv" and encode them in the same way that you encoded the paragraph vectors.

In [17]:
questions_df = pd.read_csv("questions.tsv", sep="\t")

# transform only: the IDF weights stay fitted on the paragraphs.
question_counts = hasher.transform(questions_df["question"])
question_vectors = normalize(tfidf.transform(question_counts)).toarray()
question_vectors = np.round(question_vectors, 6)

question_vectors.shape

(100, 1024)

Save your question vectors in "question-vectors.tsv" with columns question_id and question_vector_json.

In [18]:
question_vectors_df = questions_df[["question_id"]].copy()
question_vectors_df["question_vector_json"] = [json.dumps(v.tolist()) for v in question_vectors]
question_vectors_df.to_csv("question-vectors.tsv", sep="\t", index=False)

Submit "question-vectors.tsv" in Gradescope.

## Part 5: Match Questions to Paragraphs using Nearest Neighbors

Match your question vectors to paragraph vectors and identify the top 5 paragraph vectors for each question using nearest neighbors.
Specifically, use the Euclidean distance between the vectors.


In [19]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=5, metric="euclidean", algorithm="brute")
nn.fit(paragraph_vectors)

# Neighbors come back sorted nearest first, so column 0 is rank 1.
distances, neighbor_indices = nn.kneighbors(question_vectors)

neighbor_indices.shape

(100, 5)

Save your top matches in a file "question-matches.tsv" with columns question_id, question_rank, document_title, and paragraph_index.


In [20]:
match_rows = []
for q_pos, question_id in enumerate(questions_df["question_id"]):
    for rank, p_pos in enumerate(neighbor_indices[q_pos], start=1):
        match_rows.append({
            "question_id": question_id,
            "question_rank": rank,
            "document_title": parsed_df.loc[p_pos, "document_title"],
            "paragraph_index": parsed_df.loc[p_pos, "paragraph_index"],
        })

question_matches_df = pd.DataFrame(match_rows)
question_matches_df.to_csv("question-matches.tsv", sep="\t", index=False)

Submit "question-matches.tsv" in Gradescope.

## Part 6: Spot Check Question and Paragraph Matches

Review the paragraphs matched to the first 5 questions (sorted by question_id ascending).
Which paragraph was the worst match for each question?


Submit "worst-paragraphs.tsv" in Gradescope.

Write a file "worst-paragraphs.tsv" with three columns question_id, document_title, paragraph_index.

In [21]:
# Picked by reading the 5 matched paragraphs for each of the first 5 question_ids.
worst_paragraphs_df = pd.DataFrame([
    {"question_id": 1, "document_title": "Cork_(city)", "paragraph_index": 8},
    {"question_id": 4, "document_title": "BeiDou_Navigation_Satellite_System", "paragraph_index": 3},
    {"question_id": 7, "document_title": "Molotov%E2%80%93Ribbentrop_Pact", "paragraph_index": 30},
    {"question_id": 10, "document_title": "Tuvalu", "paragraph_index": 7},
    {"question_id": 13, "document_title": "American_Idol", "paragraph_index": 59},
])
worst_paragraphs_df.to_csv("worst-paragraphs.tsv", sep="\t", index=False)

## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 8: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

### Acknowledgement

No other people provided assistance. I used pandas and NumPy for data handling and vector calculations, and scikit-learn's `HashingVectorizer`, `TfidfTransformer`, and `NearestNeighbors` for the 1024-length paragraph and question vectors and the Euclidean nearest-neighbor search. GitHub Copilot Chat in Visual Studio Code assisted with interpreting the assignment, choosing and comparing the vector design, implementing the code, and checking the output files.

[Copilot Chat transcript](file:///Users/christopherabbas/Library/Application%20Support/Code/User/workspaceStorage/d690c5667068afdbf94ff162bc48cae2/GitHub.copilot-chat/transcripts/e7d90d9c-3572-4733-8668-e31c0026bd5e.jsonl)